## Transformacion de la data: KDD con Apache Spark

Se hacen las transformaciones debidas para trabajar posteriormente con la data

#### Imports

Importa las librerías de Spark y pandas que usan las celdas de selección, limpieza y transformación.

In [6]:
import os, socket
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover

pd.set_option("display.max_colwidth", 80)

#### Utilidades

Calcula la ruta del proyecto y crea la sesión de Spark.

- Constantes

In [7]:
proyecto = Path.cwd()
if not (proyecto / "parquet").exists() and (proyecto.parent / "parquet").exists():
    proyecto = proyecto.parent     # el notebook vive en 01_EDA_KDD/, el repo está un nivel arriba
parquets = proyecto / "parquet"
anios = [2023, 2024, 2025]

- Spark

In [ ]:
spark_master = os.environ.get("spark_master", "local[*]")

builder = (SparkSession.builder
           .appName("kdd")
           .master(spark_master)
           .config("spark.driver.memory",   os.environ.get("SPARK_DRIVER_MEMORY", "4g"))
           .config("spark.executor.memory", os.environ.get("SPARK_EXECUTOR_MEMORY", "4g"))
           .config("spark.sql.shuffle.partitions", "24")
           .config("spark.ui.showConsoleProgress", "false")
           .config("spark.sql.session.timeZone", "America/Lima"))

if spark_master.startswith("spark://"):
    builder = (builder.config("spark.driver.host", socket.gethostname())
                      .config("spark.driver.bindAddress", "0.0.0.0"))

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

### 1. Verificación del cluster

Confirma que el notebook está conectado al cluster de Spark del contenedor

In [9]:
sc = spark.sparkContext
estado = sc._jsc.sc().getExecutorMemoryStatus()          

print(f"Master        : {sc.master}")
print(f"Executors     : {estado.size() - 1}")
print(f"Cores totales : {sc.defaultParallelism}")
print(f"Memoria driver: {sc._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3:.1f} GB")
print("UI del job    : http://localhost:4040")

Master        : spark://spark-master:7077
Executors     : 1
Cores totales : 2
Memoria driver: 3.0 GB
UI del job    : http://localhost:4040


### 2. Cargar dataset

Lee la data cargada en `a_eda.ipynb`

In [10]:
base = spark.read.parquet(str(parquets / "staging")).cache()
N = base.count()


### 3. Áreas de análisis

**P1 — Competencia.** ¿Qué combinaciones de método, monto, rubro y región se asocian a procesos con
**un solo postor**? (baja competencia = mayor riesgo de sobrecosto)

**P2 — Fraccionamiento / procesos repetidos.** ¿Hay procesos **casi idénticos** de una misma entidad, en fechas
cercanas y con montos bajos, que sugieran dividir una compra para evadir un método más exigente?

**P3 — Precios de referencia.** Para procesos con **objeto parecido**, ¿cuánto varía el monto contratado entre
entidades y regiones?

#### 3.1 Enfoques principales


**P1 — Competencia.** ¿Qué combinaciones de método, monto, rubro y región se asocian a procesos con
**un solo postor**? (baja competencia = mayor riesgo de sobrecosto)

**P2 — Fraccionamiento / procesos repetidos.** ¿Hay procesos **casi idénticos** de una misma entidad, en fechas
cercanas y con montos bajos, que sugieran dividir una compra para evadir un método más exigente?

**P3 — Precios de referencia.** Para procesos con **objeto parecido**, ¿cuánto varía el monto contratado entre
entidades y regiones?

#### 3.2 Enfoque en técnicas

| Técnica | Aspecto a hallar | Columnas | Foco |
|---|---|---|---|
| **Jaccard + Shingling** | Similitud exacta entre descripciones en una muestra | `shingles` | P2 |
| **MinHash** | Firma compacta que aproxima Jaccard sin guardar los conjuntos completos | `shingles` | P2 |
| **LSH** (MinHashLSH) | Pares de procesos casi duplicados, y luego se filtra misma entidad + fechas cercanas | `shingles`, `entidad`, `fecha` | P2 |
| **ANN: IVF / HNSW** | Los *k* procesos más parecidos a uno dado (TF-IDF) para comparar sus montos | `tokens`, `monto_final` | P3 |
| **Bloom Filter** | ¿Este par entidad–proveedor ya apareció antes en el flujo? | `entidad`, `proveedor`, `fecha` | P2 |
| **Count-Min Sketch** | Proveedores y entidades más frecuentes (*heavy hitters*) | `proveedor`, `entidad` | P2 |
| **DGIM** | Cuántos procesos con postor único hubo en la ventana reciente del flujo | `postor_unico`, `fecha` | P1 |
| **Apriori / FP-Growth** | Reglas tipo `{CONTRATACIÓN DIRECTA, tramo alto} → {postor único}` | `canasta` | P1 |

### 4. Selección de columnas

Se queda sólo con las columnas útiles para los principales enfoques

In [11]:
cols_simples = [c for c, t in base.dtypes if not t.startswith("array")]
nulos = (base.select([(100 * F.count(F.when(F.col(c).isNull() | (F.trim(F.col(c).cast("string")) == ""), 1)) / N)
         .alias(c) for c in cols_simples]).toPandas().T[0])

descartar = sorted(set([c for c in cols_simples if nulos[c] > 95] + ["metodo", "codigo_proceso"]))
sel = base.drop(*descartar)

print("Descartadas:", descartar)
print("Seleccionadas:", sel.columns)

Descartadas: ['codigo_proceso', 'estado', 'metodo']
Seleccionadas: ['ocid', 'fecha', 'entidad', 'region', 'descripcion', 'items_desc', 'metodo_detalle', 'categoria', 'monto_referencial', 'moneda', 'n_postores', 'monto_adjudicado', 'proveedor', 'unspsc', 'anio']


Las columnas descartadas son las que se marcó con más de 95% de nulos (`estado`,`metodo`), más `codigo_proceso`, que no aportan mucho

### 5. Preprocesamiento y limpieza

Se procesa y limpia la data

#### 5.1 Auxiliares

In [12]:
def normalizar_texto(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.upper(F.trim(F.col(c))))
    return df

def quitar_duplicados(df):
    w = Window.partitionBy("ocid").orderBy(F.col("fecha").desc_nulls_last())
    return df.withColumn("_rn", F.row_number().over(w)).where("_rn = 1").drop("_rn")

def filtrar_anios(df, anios):
    return df.where(F.col("anio").isin(anios))

def filtrar_moneda_pen(df):
    return df.where(F.coalesce(F.col("moneda"), F.lit("PEN")) == "PEN").drop("moneda")

def limpiar_montos(df, columnas):
    for c in columnas:
        df = df.withColumn(c, F.when(F.col(c) > 0, F.col(c)))
    return df

def filtrar_con_descripcion(df):
    return df.where(F.length(F.trim("descripcion")) > 0)

#### 5.2 Flujo

In [13]:
pasos = [("staging", sel.count())]

limpio = normalizar_texto(sel, ["entidad", "region", "proveedor", "metodo_detalle", "categoria"])

limpio = quitar_duplicados(limpio)
pasos.append(("sin duplicados", limpio.count()))

limpio = filtrar_anios(limpio, anios)
pasos.append((f"fecha en {min(anios)}-{max(anios)}", limpio.count()))

limpio = filtrar_moneda_pen(limpio)
pasos.append(("moneda PEN", limpio.count()))

limpio = limpiar_montos(limpio, ["monto_referencial", "monto_adjudicado"])

limpio = filtrar_con_descripcion(limpio)
pasos.append(("con descripción", limpio.count()))

pd.DataFrame(pasos, columns=["paso", "filas"]).assign(perdidas=lambda d: d.filas.shift(1) - d.filas)

,paso,filas,perdidas
0,staging,236229,NaN
1,sin duplicados,236229,0.0
2,fecha en 2023-2025,235954,275.0
3,moneda PEN,231165,4789.0
4,con descripción,231165,0.0


La tabla `pasos` muestra cuántas filas se pierden en cada filtro. Duplicados y años fuera de rango pierden pocas filas

#### 5.3 Cambios

- **Duplicados:** si un `ocid` aparece más de una vez se queda el registro más reciente
- **Montos ≤ 0:** se ponen en nulo, pero el proceso se conserva para texto y frecuencias
- **Fechas:** sólo los años cargados (2023–2025) 
- **Texto** en mayúsculas y sin espacios extra
- **Moneda:** sólo PEN

### 6. Transformación

Se realizan las transformaciones debidas

#### 6.1 Auxiliares

In [14]:
con_tilde, sin_tile = "áéíóúüñ", "aeiouun"

def sin_tildes(c):
    return F.translate(F.lower(c), con_tilde, sin_tile)

def normalizar_texto_libre(df):
    return (df
            .withColumn("texto", F.concat_ws(" ", "descripcion", "items_desc"))
            .withColumn("texto", sin_tildes(F.col("texto")))
            .withColumn("texto", F.trim(F.regexp_replace(F.regexp_replace("texto", r"[^a-z0-9 ]", " "), r"\s+", " "))))

def tokenizar(df):
    stop = [s.translate(str.maketrans(con_tilde, sin_tile)) for s in StopWordsRemover.loadDefaultStopWords("spanish")]
    df = RegexTokenizer(inputCol="texto", outputCol="_tok", pattern=r"[^a-z]+", minTokenLength=3).transform(df)
    df = StopWordsRemover(inputCol="_tok", outputCol="_tok2", stopWords=stop).transform(df)
    return df.withColumn("tokens", F.array_distinct("_tok2")).drop("_tok", "_tok2")

def generar_shingles(df, k=5):
    return df.withColumn("shingles", F.when(F.length("texto") >= k,
            F.array_distinct(F.expr(f"transform(sequence(1, length(texto) - {k} + 1), i -> substring(texto, i, {k}))")))
            .otherwise(F.array("texto")))

def agregar_tramos_monto(df):
    return (df
            .withColumn("monto_final", F.coalesce("monto_adjudicado", "monto_referencial"))
            .withColumn("tramo_monto",
                F.when(F.col("monto_final").isNull(), "0_sin_monto")
                 .when(F.col("monto_final") <    50_000, "1_<50k")
                 .when(F.col("monto_final") <   200_000, "2_50k-200k")
                 .when(F.col("monto_final") < 1_000_000, "3_200k-1M")
                 .when(F.col("monto_final") < 5_000_000, "4_1M-5M")
                 .otherwise("5_>5M"))
            .withColumn("postor_unico", (F.col("n_postores") == 1).cast("int"))
            .withColumn("rubro", F.array_distinct(F.transform("unspsc", lambda x: F.substring(x, 1, 2))))
            .withColumn("mes", F.month("fecha"))
            .withColumn("periodo", F.date_format("fecha", "yyyy-MM")))

def construir_canasta(df):
    item = lambda k, c: F.when(F.col(c).isNotNull(), F.concat(F.lit(f"{k}="), F.col(c).cast("string")))
    return df.withColumn("canasta", F.array_distinct(F.concat(
            F.array_compact(F.array(
                item("cat", "categoria"), item("met", "metodo_detalle"), item("reg", "region"),
                item("monto", "tramo_monto"),
                F.when(F.col("postor_unico") == 1, F.lit("comp=postor_unico"))
                 .when(F.col("n_postores") > 1, F.lit("comp=varios_postores")))),
            F.transform("rubro", lambda r: F.concat(F.lit("rubro="), r)))))

#### 6.2 Flujo

In [15]:
t = normalizar_texto_libre(limpio)
t = tokenizar(t)
t = generar_shingles(t)
t.select("texto", "tokens", F.slice("shingles", 1, 6).alias("shingles (6 primeros)")).show(3, truncate=60)

t = agregar_tramos_monto(t)
t = construir_canasta(t)

t.groupBy("tramo_monto").agg(F.count("*").alias("procesos"), F.round(100 * F.avg("postor_unico"), 1).alias("%_postor_unico")).orderBy("tramo_monto").show()
t.select("canasta").show(3, truncate=110)

+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------+
|                                                       texto|                                                      tokens|                     shingles (6 primeros)|
+------------------------------------------------------------+------------------------------------------------------------+------------------------------------------+
|servicio de manejo forestal de la reserva ecologica del r...|[servicio, manejo, forestal, reserva, ecologica, rio, rimac]|[servi, ervic, rvici, vicio, icio , cio d]|
|servicio de arrendamiento de inmueble para el juzgado de ...|[servicio, arrendamiento, inmueble, juzgado, paz, letrado...|[servi, ervic, rvici, vicio, icio , cio d]|
|contratacion del servicio de lavado y planchado de ropa h...|[contratacion, servicio, lavado, planchado, ropa, hospita...|[contr, ontra, ntrat, trata, ratac, ataci]

`tokens` y `shingles` quedan limpios y comparables para Jaccard/MinHash/LSH y TF-IDF. El % de postor único
por tramo confirma montos bajos, menor competencia

#### 6.3 Cambios

- **`monto_final`**, **`tramo_monto`**, **`postor_unico`**, **`rubro`** (segmento UNSPSC de 2 dígitos)

- **`texto`** = `descripcion` + descripciones de los ítems en minúsculas, sin tildes ni signos

- **`canasta`**: cada proceso como transacción `clave=valor` (para Apriori / FP-Growth)

- **`shingles`**: k-shingles de 5 caracteres (Jaccard / MinHash / LSH)

- **`tokens`**: palabras sin *stopwords* en español (TF-IDF / ANN)

### 7. Dataset final

Se guarda en Parquet particionado por `anio` y `mes`: los proximos pasos leen solo lo que necesitan, y si
filtran por año o mes, Spark salta carpetas enteras sin abrirlas (*partition pruning*)

#### 7.1 Auxiliares

In [16]:
def guardar_particionado(df, columnas, path):
    final = df.select(*columnas)
    final.write.mode("overwrite").partitionBy("anio", "mes").parquet(str(path))

def verificar_dataset(path):
    procesos = spark.read.parquet(str(path))
    procesos.printSchema()
    print(f"Filas finales: {procesos.count():,} | particiones en disco: "
          f"{len(list(path.glob('anio=*/mes=*')))} carpetas anio=/mes=")
    procesos.groupBy("anio").count().orderBy("anio").show()
    return procesos

#### 7.2 Flujo

In [17]:
columnas = ["ocid", "fecha", "anio", "mes", "periodo",                   
    "entidad", "region", "proveedor",                         
    "categoria", "metodo_detalle", "n_postores", "postor_unico", 
    "monto_referencial", "monto_adjudicado", "monto_final", "tramo_monto",
    "descripcion", "texto", "tokens", "shingles",               
    "unspsc", "rubro", "canasta"]

guardar_particionado(t, columnas, parquets / "procesos")
procesos = verificar_dataset(parquets / "procesos")

procesos.where("anio = 2025 AND mes = 3").select("ocid").explain() 

root
 |-- ocid: string (nullable = true)
 |-- fecha: timestamp (nullable = true)
 |-- periodo: string (nullable = true)
 |-- entidad: string (nullable = true)
 |-- region: string (nullable = true)
 |-- proveedor: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- metodo_detalle: string (nullable = true)
 |-- n_postores: integer (nullable = true)
 |-- postor_unico: integer (nullable = true)
 |-- monto_referencial: double (nullable = true)
 |-- monto_adjudicado: double (nullable = true)
 |-- monto_final: double (nullable = true)
 |-- tramo_monto: string (nullable = true)
 |-- descripcion: string (nullable = true)
 |-- texto: string (nullable = true)
 |-- tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- shingles: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- unspsc: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- rubro: array (nullable = true)
 |    |-- element: string (cont

El plan físico debe mostrar PartitionFilters para anio/mes

### 8. Overview

#### 8.1 Proximos pasos

La data cumple los cuatro criterios necesarios, y se deja el dataset limpio en `parquet/procesos/`

| Método | Columnas |
|---|---|
| Jaccard / MinHash / LSH | `shingles` (+ `entidad`, `fecha` para filtrar pares) |
| ANN (IVF / HNSW) | `tokens` → TF-IDF, `monto_final` | III |
| Bloom / Count-Min / DGIM | `fecha` (orden), `entidad`, `proveedor`, `postor_unico` |
| Apriori / FP-Growth | `canasta` |

#### 8.2 Uso de la data

```python
procesos = spark.read.parquet("parquet/procesos")
procesos_2025 = spark.read.parquet("parquet/procesos").where("anio = 2025")
```

### 9. Cerrar sesión de Spark

Libera los recursos del cluster cerrando la SparkSession.

In [18]:
spark.stop()